## Decision Tree from Stratch

#### Libraries

In [1]:
import pandas as pd
import numpy as np

# interpretation of result
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

# reference implementation
from sklearn.tree import DecisionTreeRegressor

#### Load Case 1

In [2]:
df_train1 = pd.read_csv('../data/case1/train1.csv')
df_test1 = pd.read_csv('../data/case1/test1.csv')

##### Change relevant columns

In [3]:
# speed in krpm, n
df_train1['krpm'] = df_train1['speed_rpm'] / 1000

# pressure rise across pump
df_train1['delta_p'] = (
    df_train1['outlet_pressure_bar'] - df_train1['inlet_pressure_bar']
)

# speed in krpm, n
df_test1['krpm'] = df_test1['speed_rpm'] / 1000

# pressure rise across pump
df_test1['delta_p'] = (
    df_test1['outlet_pressure_bar'] - df_test1['inlet_pressure_bar']
)


In [4]:
exclude_cols = [
    'pump_id',
    'elapsed_s',
    'operating_state',
    'inlet_pressure_bar',
    'outlet_pressure_bar',
    'flow_m3_h',
    'speed_rpm'
]

X_train = df_train1.drop(columns = exclude_cols)
y_train = df_train1['flow_m3_h']

X_test = df_test1.drop(columns = exclude_cols)
y_test = df_test1['flow_m3_h']

In [5]:
X_train

,power_w,krpm,delta_p
0,669.992714,4.230396,1.357291
1,931.528480,3.796520,0.293043
2,724.198083,3.508264,0.185766
3,99.463217,1.498405,0.163881
4,608.195308,3.224874,0.231682
...,...,...,...
475,815.229829,4.376002,1.462426
476,515.961500,3.514587,0.927272
477,534.173942,2.935130,0.499988
478,154.677482,2.214153,0.378757


##### Define metric, that decides the best split

In [ ]:
# reference Medium: https://medium.com/@darkmortal/implementing-decision-tree-regression-from-scratch-without-using-scikit-learn-1a3ebd4b2354

def w_mse(left_labels , right_labels , y):
    '''
    Computes weighted MSE for possible splits

    Args:
        left_labels , right_labels (np.ndarray): Q values of samples sent left or right in split
        y (np.ndarray): Q values of samples in parent node

    Returns:
        weighted_mse (float): weighted MSE for split
    ''' 

    if len(left_labels) == 0 or len(right_labels) == 0:
        return np.inf

    left_mse = np.mean((left_labels - np.mean(left_labels)) ** 2)
    right_mse = np.mean((right_labels - np.mean(right_labels)) ** 2)

    weighted_mse = (len(left_labels) * left_mse + len(right_labels) * right_mse) / len(y)

    return weighted_mse

##### Implement Decision Tree Regressor from scratch

In [ ]:
# reference: exercise session ... and Medium
# with changes to fit regression problem instead of classification problem
# changes refere

class Node:
    '''
    A node in the Decision Tree Regressor

    Args:
        feature (int): index of feature to split on
        threshold (float): threshold value for the split
        left, right (node): child nodes
        value (float): predicted Q value for leaf nodes
    '''
    def __init__(self , feature = None , threshold = None , left = None , right = None , value = None):
        self.feature = feature
        self.threshold = threshold
        self.left = left
        self.right = right
        self.value = value

    def is_leaf(self):
        '''
        Return True if node is a leaf
        '''
        return self.value is not None
    
class DecisionTreeFromScratch:
    '''
    A Decision Tree Regressor from scratch

    Args:
        max_depth (int): Maximum depth of tree. Default is None, which is no depth limit
        min_samples_split (int): Minimum number of samples in a split. Default is 2
    ''' 
    def __init__(self , max_depth = None , min_samples_split = 2):
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None

    def fit(self , X , y):
        ''' 
        Build Decision Tree from data

        Args:
            X (np.ndarray): Training features
            y (np.ndarray): Target values

        Returns:
            self (DecisionTreeFromScratch): The fitted decision tree
        ''' 

        X = np.array(X)
        y = np.array(y)

        self.root = self._build_tree(X , y , depth = 0)
        return self
    
    def _build_tree(self , X , y , depth):
        ''' 
        Recursively build the Decision Tree

        Args:
            X (np.ndarray): Training features
            y (np.ndarray): Target values
            depth (int): Depth of current node

        Returns:
            Node: Subtree
        ''' 
        n_samples , n_features = X.shape

        # stopping conditions
        if (self.max_depth is not None and depth >= self.max_depth) or \
            n_samples < self.min_samples_split or len(set(y)) == 1:
                #create a leaf node with average target value
                leaf_value = np.mean(y)
                return Node(value = leaf_value)

        # find best split
        best_feature , best_threshold = self._best_split(X , y)

        # if split doesn't improve tree, create a leaf node
        if best_feature is None:
            leaf_value = np.mean(y)
            return Node(value = leaf_value)

        # split the dataset
        left_indices = X[: , best_feature] <= best_threshold
        right_indices = X[: , best_feature] > best_threshold

        # recursively build left and right subtrees
        left_child = self._build_tree(X[left_indices] , y[left_indices] , depth + 1)
        right_child = self._build_tree(X[right_indices] , y[right_indices] , depth + 1)

        return Node(
            feature = best_feature,
            threshold = best_threshold,
            left = left_child,
            right = right_child
        )
    
    def _best_split(self , X , y):
        ''' 
        Finds the best split, with weighted MSE as splitting criterion

        Args:
            X (np.ndarray): Training features
            y (np.ndarray): Target values

        Returns:
            best_feature (int): feature that creates best split
            best_threshold (float): feature value that creates best split
        '''
        n_samples , n_features = X.shape

        if n_samples <= 1:
            return None , None

        best_mse = float('inf')
        best_feature = None
        best_threshold = None

        # try all features
        for feature_idx in range(n_features):
            X_col = X[: , feature_idx]
            thresholds = np.unique(X_col)

            # try all possible thresholds
            for i in range(len(thresholds) - 1):
                threshold = (thresholds[i] + thresholds[i + 1]) / 2

                # split the data
                left_indices = X_col <= threshold
                right_indices = X_col > threshold

                left_y = y[left_indices]
                right_y = y[right_indices]

                # compute weighted MSE
                weight_mse = w_mse(left_y , right_y , y)
                

                # update best split if better
                if weight_mse < best_mse:
                    best_mse = weight_mse
                    best_feature = feature_idx
                    best_threshold = threshold

        return best_feature , best_threshold

    def predict(self , X):
        ''' 
        Predict target labels for samples in X

        Args:
            X (np.ndarray): Features to predict for

        Returns:
            np.ndarray: The predicted target values
        ''' 
        X = np.asarray(X)
        return np.array([self._traverse_tree(x , self.root) for x in X])
    
    def _traverse_tree(self , x , node):
        ''' 
        Traverse the trees to make a prediction for a single sample

        Args:
            x (np.ndarray): Feature values for single sample
            node (Node): Node to traverse from

        Returns:
            float: Predicted Q value of leaf
        ''' 
        # if we are at leaf node, return its value
        if node.is_leaf():
            return node.value

        # otherwise, traverse left or right based on feature value
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x , node.left)
        else:
            return self._traverse_tree(x , node.right)



##### Report $R^2$ and MSE for different thresholds

In [8]:
new_X_train, X_val, new_y_train, y_val = train_test_split(
    X_train, y_train, test_size = 0.2, random_state = 42
)

In [9]:
best_val_mse = float("inf")
best_val_r2 = -float("inf")
best_depth = best_split = None

for depth in [4 , 6 , 8 , 9 , 10 , 11 , 12]:
    for splits in [2 , 3 , 4 , 5 , 10 , 20]:
        dt = DecisionTreeFromScratch(
            max_depth = depth,
            min_samples_split = splits
        )

        dt.fit(new_X_train, new_y_train)

        y_train_pred = dt.predict(new_X_train)
        y_val_pred = dt.predict(X_val)

        train_mse = mean_squared_error(new_y_train , y_train_pred)
        val_mse = mean_squared_error(y_val , y_val_pred)

        train_r2 = r2_score(new_y_train, y_train_pred)
        val_r2 = r2_score(y_val, y_val_pred)

        if val_mse < best_val_mse:
            best_split = splits
            best_depth = depth
            best_val_mse = val_mse
            best_val_r2 = val_r2

        # print(
        #     f"Depth = {depth} and, "
        #     f"Minimum samples = {splits}: "
        #     f"Train MSE = {train_mse:.3f}, "
        #     f"Validation MSE = {val_mse:.3f}, "
        #     f"Train R2 = {train_r2:.3f}, "
        #     f"Validation R2 = {val_r2:.3f}"
        # )

print(
    f"Best depth = {best_depth} and, "
    f"Best minimum samples = {best_split}: "
    f"Best validation MSE = {best_val_mse:.3f}, "
    f"Best validation R2 = {best_val_r2:.3f}"
)

Best depth = 10 and, Best minimum samples = 2: Best validation MSE = 7.183, Best validation R2 = 0.952


##### Refence implementation of Decision Tree using scikit 

In [10]:
best_val_mse_ref = float("inf")
best_val_r2_ref = -float("inf")
best_depth_ref = best_split_ref = None


for depth in [4 , 6 , 8 , 9 , 10 , 11 , 12]:
    for splits in [2 , 3 , 4 , 5]:
        regressor = DecisionTreeRegressor(
            max_depth = depth, 
            min_samples_split = splits, 
            random_state = 42)
        
        regressor.fit(new_X_train , new_y_train)

        y_train_pred_ref = regressor.predict(new_X_train)
        y_val_pred_ref = regressor.predict(X_val)

        train_mse_ref = mean_squared_error(new_y_train , y_train_pred_ref)
        val_mse_ref = mean_squared_error(y_val , y_val_pred_ref)

        train_r2_ref = r2_score(new_y_train, y_train_pred_ref)
        val_r2_ref = r2_score(y_val, y_val_pred_ref)

        if val_r2_ref > best_val_r2_ref:
            best_split_ref = splits
            best_depth_ref = depth
            best_val_mse_ref = val_mse_ref
            best_val_r2_ref = val_r2_ref

        # print(
        #     f"Depth = {depth} and, "
        #     f"Minimum samples = {splits}: "
        #     f"Train MSE = {train_mse_ref:.3f}, "
        #     f"Validation MSE = {val_mse_ref:.3f}, "
        #     f"Train R2 = {train_r2_ref:.3f}, "
        #     f"Validation R2 = {val_r2_ref:.3f}"
        # )

print(
    f"Best depth = {best_depth_ref} and, "
    f"Best minimum samples = {best_split_ref}: "
    f"Best validation MSE = {best_val_mse_ref:.3f}, "
    f"Best validation R2 = {best_val_r2_ref:.3f}"
)

Best depth = 10 and, Best minimum samples = 3: Best validation MSE = 7.207, Best validation R2 = 0.952


##### Compare final reference and scratch implementation

In [11]:
scratch_final = DecisionTreeFromScratch(
    max_depth = best_depth,
    min_samples_split = best_split
).fit(X_train, y_train)

scratch_y_test_pred = scratch_final.predict(X_test)
print(f"Test MSE for scratch implementation = {mean_squared_error(y_test, scratch_y_test_pred):.3f}, "
      f"Test R2 for scratch implementation = {r2_score(y_test, scratch_y_test_pred):.3f}")

ref_final = DecisionTreeRegressor(
    max_depth = best_depth_ref,
    min_samples_split = best_split_ref,
    random_state = 42
).fit(X_train, y_train)

ref_y_test_pred = ref_final.predict(X_test)
print(f"Test MSE for reference implementation = {mean_squared_error(y_test, ref_y_test_pred):.3f}, "
      f"Test R2 for refrence implementation = {r2_score(y_test, ref_y_test_pred):.3f}")

Test MSE for scratch implementation = 6.099, Test R2 for scratch implementation = 0.958
Test MSE for reference implementation = 6.673, Test R2 for refrence implementation = 0.954
